In [1]:
# pip install keras-tuner

In [3]:
import pandas as pd
import tensorflow
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [4]:
data = pd.read_csv('huge_1M_titanic.csv')

In [5]:
data = data.sample(10000, random_state=42)

In [6]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
987231,988541,0,3,"Name988541, Mr. Surname988541",male,13.0,5,2,315084,44.567495,B57 B59 B63 B66,S
79954,81264,1,1,"Name81264, Miss. Surname81264",female,73.0,1,1,367655,133.025378,NaN,S
567130,568440,0,3,"Name568440, Mr. Surname568440",male,35.0,0,0,C.A. 5547,6.131359,NaN,S
500891,502201,1,1,"Name502201, Mrs. Surname502201",female,45.0,0,0,345764,141.897841,NaN,C
55399,56709,0,3,"Name56709, Mr. Surname56709",male,34.0,0,0,345774,19.546819,NaN,S


In [7]:
data = data.drop(columns = ['PassengerId','Name','Age','Ticket','Cabin'])

In [8]:
data['Embarked'] = data['Embarked'].replace({'S':'Southampton','C':'Chebourg','Q':'Queenstown'})

In [9]:
data.dropna(subset=['Embarked'],inplace = True)

In [10]:
data['Fare'] = data['Fare'].astype('int')

In [11]:
label = LabelEncoder()
onehot = OneHotEncoder(sparse_output = False)

In [12]:
data['Sex'] = label.fit_transform(data['Sex'])

In [13]:
Embarked = onehot.fit_transform(data[['Embarked']])
Embarked = pd.DataFrame(Embarked, columns = onehot.get_feature_names_out())
data = pd.concat([data.drop(columns = ['Embarked']),Embarked], axis=1)

In [14]:
scale = StandardScaler()

In [15]:
num_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']
data[num_cols] = scale.fit_transform(data[num_cols])

In [16]:
data = data.dropna()

In [17]:
X = data.drop(columns = ['Survived'])
y = data['Survived']

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 20, random_state = 42)

# HyperParameter Tuning
### 1--> No. of Hidden Layers
### 2--> No. of Nodes in the Hidden Layer
### 3--> Optimal Optimizer
### 4--> Optimal Dropout Layer Value

In [1]:
def build_model(hp):

    optimizer = hp.Choice('optimizer',values = ['SGD', 'RMSprop', 'Adam', 'AdamW', 'Adadelta', 'Adagrad', 'Adamax']) #HP Optimal Optimizer Value
    model = Sequential()
    model.add(Input(shape = (X_train.shape[1],)))
    for i in range(hp.Int('hidden',min_value = 1, max_value = 10, step = 1)): #HP No. of Optimal Hidden Layers
        nodes = hp.Int('nodes',min_value = 8, max_value = 128, step = 8) #HP No. of Optimal Nodes in Hidden Layer
        model.add(Dense(nodes, activation = 'relu'))
        
        dropout_val = hp.Float('dropout',min_value = 0.1, max_value = 0.9, step = 0.1) #HP Optimal Dropout Value
        model.add(Dropout(dropout_val))
    model.add(Dense(1,activation = 'sigmoid'))

    model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

    return model

In [19]:
tuner = kt.RandomSearch(build_model,objective = 'val_accuracy', max_trials = 5,directory = 'finalHP')

In [20]:
tuner.search(X_train,y_train, validation_data= (X_test,y_test),epochs = 20)

Trial 5 Complete [00h 00m 05s]
val_accuracy: 0.6499999761581421

Best val_accuracy So Far: 0.949999988079071
Total elapsed time: 00h 00m 27s


In [21]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'Adadelta', 'hidden': 7, 'nodes': 120, 'dropout': 0.2}

In [22]:
model = tuner.get_best_models(num_models = 1)[0]

c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adadelta', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [23]:
model.fit(X_train,y_train, validation_data = (X_test,y_test), epochs = 100,batch_size = 32)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 123ms/step - accuracy: 0.4524 - loss: 0.6985 - val_accuracy: 0.9500 - val_loss: 0.6896
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5000 - loss: 0.6915 - val_accuracy: 0.9500 - val_loss: 0.6896
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4405 - loss: 0.6954 - val_accuracy: 0.9500 - val_loss: 0.6896
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4405 - loss: 0.6968 - val_accuracy: 0.9000 - val_loss: 0.6896
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4405 - loss: 0.6960 - val_accuracy: 0.9000 - val_loss: 0.6896
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4405 - loss: 0.7000 - val_accuracy: 0.9000 - val_loss: 0.6896
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4286 - loss: 0.6936 - val_accuracy: 0.9000 - val_loss: 0.6896
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4881 - loss: 0.6944 - val_accuracy: 0.8500 - val_loss